In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import adjusted_rand_score as ari_score
from sklearn.metrics import silhouette_score
from scipy.stats import spearmanr

# -----------------------------
# Settings
# -----------------------------
sc.settings.verbosity = 0
sns.set(style="whitegrid")

ground_truth_path = "Data/adata_raw_qc.h5ad"

# Folders for each method
imputed_dirs = {
    "SoftImpute": "imputed_softimpute",
    "MAGIC": "imputed_h5ad",
    "KNN": "imputed_knn",
    "Mean": "imputed_mean",
    "GAN": "imputed_gan",
    "Iterative": "imputed_iterative"
}

# Define marker genes (from your list)
marker_genes = [
    "CD34", "SPINK2", "SLAMF7", "CD27", "PF4","PPBP","CD79B","CD79A","MS4A1","CD19","CD72","CD40","CD22","NKG7",
                "GZMB","FGFBP2","SPON2","KLRC1","KLRD1","MS4A7","FCN1","S100A12","S100A9","S100A8","FCGR3A","CD14","LDHB","CD4",
                "CD8B","CD8A","CD3D","CD3E","CD3G"]

# -----------------------------
# Helper functions
# -----------------------------
def preprocess_adata(adata):
    """Basic preprocessing to ensure clean AnnData."""
    adata.var_names_make_unique()
    adata.obs_names_make_unique()
    adata.X = np.nan_to_num(adata.X, nan=0.0, posinf=0.0, neginf=0.0)
    return adata


def compute_silhouette(adata):
    """Compute silhouette score from PCA representation & leiden labels."""
    if "X_pca" not in adata.obsm or "leiden" not in adata.obs:
        return np.nan
    try:
        score = silhouette_score(adata.obsm["X_pca"], adata.obs["leiden"])
    except Exception:
        score = np.nan
    return score


def compute_gene_correlation(adata_gt, adata_imp):
    """Compute gene-wise Spearman correlation between ground truth and imputed."""
    common_genes = adata_gt.var_names.intersection(adata_imp.var_names)
    if len(common_genes) == 0:
        return np.nan
    
    gt_expr = adata_gt[:, common_genes].X.toarray() if not isinstance(adata_gt.X, np.ndarray) else adata_gt[:, common_genes].X
    imp_expr = adata_imp[:, common_genes].X.toarray() if not isinstance(adata_imp.X, np.ndarray) else adata_imp[:, common_genes].X
    
    corrs = []
    for g in range(len(common_genes)):
        rho, _ = spearmanr(gt_expr[:, g], imp_expr[:, g])
        corrs.append(rho)
    return np.nanmean(corrs)


def check_marker_gene_preservation(adata_gt, adata_imp, markers):
    """Compare average expression of marker genes between GT and imputed."""
    preserved = {}
    for gene in markers:
        if gene not in adata_gt.var_names or gene not in adata_imp.var_names:
            preserved[gene] = np.nan
            continue
        mean_gt = np.mean(adata_gt[:, gene].X)
        mean_imp = np.mean(adata_imp[:, gene].X)
        # Spearman correlation is not defined for single values → fallback to ratio
        if mean_gt > 0 and mean_imp > 0:
            preserved[gene] = min(mean_gt, mean_imp) / max(mean_gt, mean_imp)
        else:
            preserved[gene] = np.nan
    return preserved

# -----------------------------
# Load ground truth
# -----------------------------
print(" Preprocessing ground truth...")
adata_gt = sc.read_h5ad(ground_truth_path)
adata_gt = preprocess_adata(adata_gt)

# -----------------------------
# Evaluate all methods
# -----------------------------
results = []

for method, imputed_dir in imputed_dirs.items():
    if not os.path.exists(imputed_dir):
        print(f"Skipping {method}, folder not found: {imputed_dir}")
        continue

    for fname in os.listdir(imputed_dir):
        if not fname.endswith(".h5ad"):
            continue

        fpath = os.path.join(imputed_dir, fname)
        try:
            adata_imp = sc.read_h5ad(fpath)
            adata_imp = preprocess_adata(adata_imp)

            # Align cells
            common_cells = adata_gt.obs_names.intersection(adata_imp.obs_names)
            adata_gt_sub = adata_gt[common_cells]
            adata_imp_sub = adata_imp[common_cells]

            # --- Metrics ---
            ari = ari_score(adata_gt_sub.obs["leiden"], adata_imp_sub.obs["leiden"]) if "leiden" in adata_gt_sub.obs and "leiden" in adata_imp_sub.obs else np.nan
            sil = compute_silhouette(adata_imp_sub)
            gene_corr = compute_gene_correlation(adata_gt_sub, adata_imp_sub)
            marker_corrs = check_marker_gene_preservation(adata_gt_sub, adata_imp_sub, marker_genes)

            # Parse metadata
            mf, run = None, None
            for part in fname.split("_"):
                if part.startswith("mf"):
                    mf = float(part.replace("mf", "").replace(".h5ad", "")) / 100
                if part.startswith("run"):
                    run = int(part.replace("run", "").replace(".h5ad", ""))

            res = {
                "file": fname,
                "method": method,
                "ARI": ari,
                "Silhouette": sil,
                "GeneCorr": gene_corr,
                "missing_fraction": mf,
                "run": run
            }
            # Add marker gene results
            for g, val in marker_corrs.items():
                res[f"Marker_{g}"] = val

            results.append(res)

            print(f"✅ {method} - {fname}: ARI={ari:.3f}, Sil={sil:.3f}, GeneCorr={gene_corr:.3f}")

        except Exception as e:
            print(f"❌ Error with {method} - {fname}: {e}")

# -----------------------------
# Results DataFrame
# -----------------------------
results_df = pd.DataFrame(results)
results_df.to_csv("evaluation_marker_genes.csv", index=False)

print("\n✅ Saved results to evaluation_marker_genes.csv")

# -----------------------------
# Heatmap of Marker Gene Preservation
# -----------------------------
marker_cols = [c for c in results_df.columns if c.startswith("Marker_")]

if len(marker_cols) == 0:
    print("⚠️ No marker gene results found in results_df")
else:
    marker_data = results_df.groupby("method")[marker_cols].mean().T
    marker_data.index = [g.replace("Marker_", "") for g in marker_data.index]

    plt.figure(figsize=(12, 8))
    sns.heatmap(
        marker_data,
        annot=True,
        fmt=".2f",
        cmap="viridis",
        cbar_kws={'label': 'Marker Gene Preservation'}
    )
    plt.title("Marker Gene Preservation (Mean across missing fractions & runs)")
    plt.ylabel("Marker Genes")
    plt.xlabel("Imputation Method")
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# -----------------------------
# Load results
# -----------------------------
results_df = pd.read_csv("evaluation_marker_genes.csv")

# Extract marker gene columns only
marker_cols = [c for c in results_df.columns if c.startswith("Marker_")]

# Aggregate across runs & missing fractions
marker_data = (
    results_df.groupby("method")[marker_cols]
    .mean()
    .T
)

# Clean column names
marker_data.index = [g.replace("Marker_", "") for g in marker_data.index]

# Drop genes missing everywhere
marker_data = marker_data.dropna(how="all")

# Reorder methods for consistency (publication style)
method_order = ["SoftImpute", "MAGIC", "KNN", "Mean", "GAN", "Iterative"]
marker_data = marker_data[method_order]

# Optional: order genes by mean preservation
gene_order = marker_data.mean(axis=1).sort_values(ascending=False).index
marker_data = marker_data.loc[gene_order]

# -----------------------------
# Heatmap (publication ready)
# -----------------------------
plt.figure(figsize=(12, 10))

ax = sns.heatmap(
    marker_data,
    cmap="coolwarm",
    vmin=0, vmax=1,
    cbar_kws={'label': 'Marker Gene Preservation Score'},
    linewidths=0.3,
    linecolor="gray",
    annot=True, fmt=".2f", annot_kws={"size": 7},  # optional: show exact values
)

# Labels & titles
ax.set_title("Marker Gene Preservation Across Imputation Methods", fontsize=16, pad=20)
ax.set_xlabel("Imputation Method", fontsize=14)
ax.set_ylabel("Marker Genes", fontsize=14)

# Tick labels formatting
ax.set_xticklabels(ax.get_xticklabels(), fontsize=12, rotation=30, ha="right")
ax.set_yticklabels(ax.get_yticklabels(), fontsize=11)

plt.tight_layout()
plt.show()
